# Scaling Laws: How Big Should the Model Be? How Much Data? How Much Money?

> **Background**: you want to train an LLM. Your boss gives you a $1M GPU budget and asks:
> "With this budget, how big should the model be? How much data should we use? How good can it get?"
>
> Goal for this part: **understand the three paradigm shifts of scaling laws, and learn to estimate training compute and VRAM.**

There are only two core questions:
1. **Given a compute budget, what is the best model-vs-data tradeoff?** -> scaling laws
2. **To actually train a model, how many GPUs and how much money do we need?** -> FLOPs / VRAM estimation

We will use small, high-school-level examples and hand calculations so every step is concrete.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Part A: Scaling laws

### 1. What is a power law? Understand it via savings interest

Before scaling laws, you must understand the idea of a **power law**.

#### 1.1 Linear improvement vs power-law improvement

Suppose your saved money doubles each month:

- **Linear**: every doubling adds a fixed amount
  - 100 -> gain 10
  - 200 -> gain 20 (+10)
  - 400 -> gain 30 (+10)

- **Power law**: every doubling improves by a fixed *ratio*
  - 100 -> gain 10
  - 200 -> gain 18 (+80%)
  - 400 -> gain 32.4 (+80%)

LLM training loss often follows a power law: **when model size doubles, loss drops by roughly a fixed ratio.**

A common form is:
`Loss(N) ≈ a × N^(-b) + c`

Where:
- `N` = number of parameters
- `a, b, c` = constants fit from experiments
- `N^(-b)` gets smaller as `N` grows
- `c` is the irreducible loss floor (you cannot go below it by scaling N)


In [ ]:
# === power lawdiminishing的manualdemo ===
print("=== power lawdiminishing：Loss 随Model size变化 ===")
print()

# 模拟 loss = a * N^(-b) + c
# 用很小的数字来demo
a, b_pow, c = 5.0, 0.05, 2.0

# model从 1M params到 100B params
sizes = [1, 10, 100, 1000, 10000, 100000]  # 单位：百万params
labels = ["1M", "10M", "100M", "1B", "10B", "100B"]

print(f"公式: Loss = {a} × N^(-{b_pow}) + {c}")
print()
print(f"{'Model size':>10s} {'Loss':>8s} {'相比上一级':>12s}")
print("-" * 32)

prev_loss = None
for size, label in zip(sizes, labels):
    N = size * 1e6  # 转成params数量
    loss = a * N**(-b_pow) + c
    
    if prev_loss:
        improvement = (prev_loss - loss) / prev_loss * 100
        print(f"{label:>10s}  {loss:>6.2f}  降低 {improvement:>5.1f}%")
    else:
        print(f"{label:>10s}  {loss:>6.2f}   —")
    prev_loss = loss

print()
print("Key observation：")
print("  1. 每次model扩大 10 倍,loss 确实在降")
print("  2. butdrop的幅度smaller and smaller -> 边际gaindiminishing")
print("  3. 从 1B -> 10B drop的幅度 < 从 1M -> 10M")
print()
print("这就是power law的核心Conclusion：bigger helps,but越来越less worth it.")
print("youneed一个公式来告诉you：到哪一步就less worth it得再大了.")

### 2. Kaplan scaling law (OpenAI, 2020): "scale the model first"

In 2020, OpenAI ran many experiments to answer:
Given a fixed compute budget `C` (e.g. `1e20` FLOPs), what is the best choice of model size `N` and data size `D`?

#### 2.1 Key finding

The optimal scaling they reported looked like:

```
N_opt ∝ C^0.73    (model size grows fast with compute)
D_opt ∝ C^0.27    (data grows slowly with compute)
```

Because `0.73 > 0.27`, **model size grows much faster than data**.

Plain English: when you double the compute budget, you should spend most of the extra budget on a larger model, and a smaller part on more data.

#### 2.2 Feel it with numbers


In [ ]:
# === Kaplan 定律manual ===
print("=== Kaplan Scaling laws：给定Compute budget,optimal配置是什么？ ===")
print()

# Reference point：assume C=1e20 FLOPs 时,N=1B, D=20B tokens
N_ref = 1e9   # 10 亿params
D_ref = 20e9  # 200 亿 token
C_ref = 1e20  # Compute budget

print(f"参照点: C={C_ref/1e20:.0f}×10²⁰ FLOPs -> N={N_ref/1e9:.0f}B, D={D_ref/1e9:.0f}B tokens")
print()

# Compute budgetdoubling
budgets = [1e20, 2e20, 4e20, 8e20, 1e21, 1e22]
budget_labels = ["1×", "2×", "4×", "8×", "10×", "100×"]

print(f"{'Compute budget':>12s}  {'Model size(Kaplan)':>18s}  {'Data size(Kaplan)':>18s}  {'D/N比':>10s}")
print("-" * 64)

for C, label in zip(budgets, budget_labels):
    ratio = C / C_ref
    N_opt = N_ref * (ratio ** 0.73)
    D_opt = D_ref * (ratio ** 0.27)
    dn_ratio = D_opt / N_opt
    
    if N_opt >= 1e9:
        n_str = f"{N_opt/1e9:.1f}B"
    else:
        n_str = f"{N_opt/1e6:.0f}M"
    
    d_str = f"{D_opt/1e9:.1f}B tokens"
    print(f"{label:>12s}  {n_str:>18s}  {d_str:>18s}  {dn_ratio:>8.1f}x")

print()
print("Kaplan 的判断：")
print("  • Compute budget越大,model越大(远超数据增长)")
print("  • D/N 比smaller and smaller -> 大model可以「省着用」数据")
print("  • 比如 10B model只need 40B tokens(而不是直觉上的越多越好)")

### 3. Chinchilla scaling law (DeepMind, 2022): "model and data both matter"

#### 3.1 Why Kaplan was "wrong"

DeepMind repeated the experiments and found that in OpenAI's earlier setting, when compute increased, **data did not increase enough** (high-quality data was limited at the time).
So "scale the model first" was a conclusion under a *data-starved* regime.

#### 3.2 Chinchilla's conclusion

DeepMind swept many (N, D) combinations for each compute budget and found the true optimum:

```
N_opt ∝ C^0.5
D_opt ∝ C^0.5
-> model and data are equally important
-> D_opt ≈ 20 × N_opt   (about 20 tokens per parameter)
```

#### 3.3 Under the same compute budget, Kaplan vs Chinchilla choose different points


In [ ]:
# === Kaplan vs Chinchilla 对决 ===
print("=== Kaplan vs Chinchilla：同一预算,不同方案 ===")
print()

# 一个具体的预算
C = 1e23  # 约 GPT-3 的training量级
ratio = C / C_ref

N_k = N_ref * (ratio ** 0.73)
D_k = D_ref * (ratio ** 0.27)

N_c = N_ref * (ratio ** 0.5)
D_c = D_ref * (ratio ** 0.5)

print(f"计算预算: {C/1e23:.0f} × 10²³ FLOPs")
print()
print(f"{'':>20s}  {'Model size':>12s}  {'Data size':>14s}  {'D/N比':>10s}")
print("-" * 60)
print(f"{'Kaplan 说':>20s}  {N_k/1e9:>8.1f}B    {D_k/1e9:>9.1f}B tokens  {D_k/N_k:>8.1f}x")
print(f"{'Chinchilla 说':>20s}  {N_c/1e9:>8.1f}B    {D_c/1e9:>9.1f}B tokens  {D_c/N_c:>8.1f}x")
print()

print(f"对比:")
print(f"  Kaplan 方案:   模型大 {N_k/N_c:.1f}×，但数据只有 Chinchilla 的 {D_k/D_c:.1f}×")
print(f"  Chinchilla 方案: 模型更小，但数据更多")
print()
print(f"实验结果: Chinchilla 方案 loss 更低 -> 推翻了 Kaplan")
print(f"          -> 数据少的大模型 < 数据多的中等模型")

#### 3.4 The shocking example

DeepMind trained two models under roughly the same compute budget:

```
Gopher:      280B params, 300B tokens   (Kaplan-style)
Chinchilla:   70B params, 1.4T tokens  (Chinchilla-style)
  same total compute

Result: Chinchilla (70B) >> Gopher (280B)
        -> 4x smaller model, but 4.7x more data, and it wins
```

The punchline: many large models in that era were likely **under-trained on data**.


### 4. Post-Chinchilla: "overtraining" can be even better

Chinchilla says D/N ~= 20 is optimal, but 2023-2024 practice often goes far beyond that.

#### 4.1 Why? Because Chinchilla optimizes *training cost*

Chinchilla's "optimal" means: **under a fixed training compute budget, minimize loss.**

But in deployment, **inference cost dominates**. Serving millions of users can cost far more than one-time training.

So the strategy becomes: **train a smaller model, feed it a lot more data, and get strong performance with cheaper inference.**

Example:

```
LLaMA 3 8B:
  Chinchilla would suggest ~ 8B * 20 = 160B tokens
  In practice: ~15T tokens (about 94x the Chinchilla point)
  Result: much stronger than an 8B model trained at the Chinchilla optimum
```

This is the train-vs-infer tradeoff: pay more once in training to save a lot in inference.


In [ ]:
# === 实际model的 D/N 比 ===
print("=== major models的 D/N 比 ===")
print()

real_models = [
    ("Chinchilla (optimal)", 70, 1400, "20x"),
    ("LLaMA 7B", 7, 1000, "143x"),
    ("LLaMA 2 7B", 7, 2000, "286x"),
    ("LLaMA 3 8B", 8, 15000, "1875x"),
    ("DeepSeek-V2", 236, 8100, "34x (active: ~386x)"),
]

print(f"{'model':<20s} {'params':>8s} {'Data size':>12s} {'D/N':>10s}")
print("-" * 54)
for name, params_b, tokens_b, dn in real_models:
    print(f"{name:<20s} {params_b:>5}B   {tokens_b:>6}B tokens  {dn:>10s}")

print()
print("观察: 2023 年后的model D/N 比远超 20x")
print("  -> industryconsensus已变成「小model + 海量数据」")
print("  -> 因为inference时用小model便宜得多,training时多花点数据成本也值得")

### 5. µP (Maximal Update Parameterization): how to tune hyperparams on small models

#### 5.1 The problem

Before training a 70B model, you want to tune hyperparameters (learning rate, init, etc.).
But you cannot grid-search on 70B; one run can cost hundreds of thousands.

A common workflow is: tune on a 10M model, then reuse the hyperparams on 70B.

**Problem**: it often does not transfer. The best LR on 10M can be unstable on 70B.

#### 5.2 Why?

Different model sizes have different activation and gradient scales.
An LR that is right for 10M can be too small (no learning) or too large (diverges) for 70B.

#### 5.3 How µP helps

µP rescales initialization variance and learning rates so that **early training dynamics match across model sizes**.

Plain English: µP makes 10M and 70B models "feel" the same to train, so hyperparams found on small scale can transfer.

```
Without µP: tune lr=0.01 on 10M -> move to 70B -> unstable, retune
With µP:    tune lr=0.01 on 10M -> move to 70B -> works
```

This can save huge human and compute cost.


---

## Part B: Resource estimation

### 6. FLOPs estimation: how much compute does training need?

#### 6.1 Core formula

```
C ≈ 6 × P × D
```

Where:
- `C` = total compute (FLOPs)
- `P` = number of parameters
- `D` = number of training tokens

#### 6.2 Why the factor 6? Do it by hand

Training one token needs forward + backward.

Forward pass:
- dominated by matrix multiplies
- roughly ~2P FLOPs per token

Backward pass:
- computing gradients is roughly ~2x forward
- ~4P FLOPs per token

Total: `2P + 4P = 6P` FLOPs per token.
So for D tokens: `C = 6PD`.

#### 6.3 Compute a concrete example


In [ ]:
# === FLOPs estimation：一步一步算 ===
print("=== FLOPs estimation：manual ===")
print()

# Question：训一个 LLaMA 7B,用了 1T tokens,need多少 FLOPs？
P = 7e9     # 7B params
D = 1e12    # 1T tokens

print(f"模型参数 P = {P/1e9:.0f}B")
print(f"训练数据 D = {D/1e12:.0f}T tokens")
print()

# Step 1: 每个 token 的 FLOPs
flops_per_token = 6 * P
print(f"Step 1 — 每个 token 的 FLOPs = 6 × {P/1e9:.0f}B")
print(f"         = 6 × {P:.1e}")
print(f"         = {flops_per_token:.1e} FLOPs/token")
print()

# Step 2: 总 FLOPs
total_flops = flops_per_token * D
print(f"Step 2 — 总 FLOPs = {flops_per_token:.1e} × {D/1e12:.0f}T")
print(f"         = {flops_per_token:.1e} × {D:.1e}")
print(f"         = {total_flops:.1e} FLOPs")
print()

# Convert to human-readable
def format_flops(flops):
    if flops >= 1e24:
        return f"{flops/1e24:.1f} YFLOPs (1e24)"
    elif flops >= 1e21:
        return f"{flops/1e21:.1f} ZFLOPs (1e21)"
    elif flops >= 1e18:
        return f"{flops/1e18:.1f} EFLOPs (1e18)"
    else:
        return f"{flops/1e15:.1f} PFLOPs (1e15)"

print(f"Step 3 — 结果: {format_flops(total_flops)}")
print()

# Step 4: need多少 GPU hours？
print(f"Step 4 — 用 A100 (312 TFLOPS FP16, 利用率 50%):")
effective_tflops = 312 * 0.5  # 有效算力
flops_per_second = effective_tflops * 1e12
seconds = total_flops / flops_per_second
hours = seconds / 3600
days = hours / 24
print(f"  有效算力: {effective_tflops:.0f} TFLOPS")
print(f"  需要时间: {hours:,.0f} 小时 = {days:.0f} 天 (单卡)")
print(f"  如果用 256 张 A100: {hours/256:.0f} 小时 = {days/256:.1f} 天")

In [ ]:
# === major modelstraining FLOPs Compare ===
print("=== 主流modeltraining FLOPs overview ===")
print()

models = [
    ("GPT-3 175B", 175e9, 300e9),
    ("LLaMA 7B", 7e9, 1e12),
    ("LLaMA 65B", 65e9, 1.4e12),
    ("Chinchilla 70B", 70e9, 1.4e12),
    ("LLaMA 2 70B", 70e9, 2e12),
    ("LLaMA 3 8B", 8e9, 15e12),
    ("LLaMA 3 70B", 70e9, 15e12),
    ("DeepSeek-V3 (total)", 671e9, 14.8e12),
]

print(f"{'model':<20s} {'params':>8s} {'Data size':>12s} {'总FLOPs':>14s} {'A100·days(256GPU)':>16s}")
print("-" * 76)

for name, params, tokens in models:
    flops = 6 * params * tokens
    effective_tflops = 312 * 0.5
    hours = flops / (effective_tflops * 1e12 * 3600)
    days_256 = hours / 256 / 24
    
    print(f"{name:<20s} {params/1e9:>5.0f}B  {tokens/1e9:>8.0f}B tokens"
          f"  {format_flops(flops):>14s}  {days_256:>12.1f} 天")

print()
print("Note:")
print("  • This is only an estimate(In practice there are alsoactivations、communication等overhead,约 +10-20%)")
print("  • utilization 50% 是乐观估计,大集群communication可能更低")
print("  • 以上是单次training的成本,does not includefailed reruns、tuning experiments等")

### 7. VRAM estimation: how many GPUs do you need?

#### 7.1 What consumes VRAM during training?

During training, VRAM has several major tenants:

```
+-----------------------------------------+
|              GPU VRAM                   |
+-----------------------------------------+
| 1) Parameters (FP16)           2P bytes |
| 2) Gradients (FP16)            2P bytes |
| 3) AdamW states (FP32)         8P bytes |
|    - m (momentum)              4P bytes |
|    - v (variance)              4P bytes |
| 4) Master params (FP32)        4P bytes |
| 5) Activations                 ~2-10P   |
+-----------------------------------------+
| Total                         ~18-26P   |
+-----------------------------------------+
```

Rule of thumb: **~20 bytes of VRAM per parameter** for training.

#### 7.2 Example: LLaMA 7B


In [ ]:
# === VRAMestimation：manual ===
print("=== VRAMestimation：LLaMA 7B trainingneed多少VRAM？ ===")
print()

P = 7e9  # 7B params

# 逐项计算
param_fp16 = 2 * P
grad_fp16 = 2 * P
adam_m = 4 * P
adam_v = 4 * P
param_fp32 = 4 * P

model_optimizer = param_fp16 + grad_fp16 + adam_m + adam_v + param_fp32

print(f"模型参数 + 优化器状态：")
print(f"  ① 模型参数 (FP16): {param_fp16/1e9:.1f} GB")
print(f"  ② 梯度 (FP16):     {grad_fp16/1e9:.1f} GB")
print(f"  ③ Adam m (FP32):   {adam_m/1e9:.1f} GB")
print(f"  ④ Adam v (FP32):   {adam_v/1e9:.1f} GB")
print(f"  ⑤ 参数备份 (FP32):  {param_fp32/1e9:.1f} GB")
print(f"  小计:              {model_optimizer/1e9:.1f} GB")
print(f"  = {model_optimizer/P:.0f} bytes/param")
print()

# activationsestimation(粗略)
# activations取决于 batch_size、seq_len、d_model、num_layers
batch_size = 4
seq_len = 4096

print(f"激活值（取决于 batch={batch_size}, seq_len={seq_len}）：")
print(f"  粗略估计: 约 2-10P = {2*P/1e9:.0f}-{10*P/1e9:.0f} GB")
print()

# 总计
activation_low = 2 * P
activation_high = 10 * P
total_low = (model_optimizer + activation_low) / 1e9
total_high = (model_optimizer + activation_high) / 1e9

print(f"总显存: {total_low:.0f} - {total_high:.0f} GB")
print(f"A100 (80G) 需要: {total_low/80:.0f} - {total_high/80:.0f} 张")
print()

# 几个典型modelneed的VRAM
print("=== 各modeltrainingVRAM需求(estimation)===")
print()
print(f"{'model':<18s} {'model+optimizer':>12s} {'总VRAM(含激活)':>16s} {'A100(80G)':>12s}")
print("-" * 62)

for name, params, _ in [
    ("1.5B 小model", 1.5e9, None),
    ("LLaMA 7B", 7e9, None),
    ("LLaMA 13B", 13e9, None),
    ("LLaMA 65B", 65e9, None),
    ("GPT-3 175B", 175e9, None),
]:
    m_o = 16 * params / 1e9
    total = 20 * params / 1e9  # 经验法则 ~20 bytes/param
    gpus = total / 80
    print(f"{name:<18s} {m_o:>8.0f} GB   ~{total:>10.0f} GB      {gpus:>8.0f} 张")

print()
print("经验法则: 1B params ≈ 20GB VRAM (training时)")
print("inference只need ~2GB/B(do not needgradients、optimizer、activations)")

### 8. Summary of the three paradigm shifts

```
2020 Kaplan (OpenAI):
  "Scale the model; data can be smaller"
  N ∝ C^0.73, D ∝ C^0.27
  -> GPT-3 175B + 300B tokens (D/N=1.7x)

2022 Chinchilla (DeepMind):
  "Model and data are equally important"
  N ∝ C^0.5, D ∝ C^0.5, D/N ≈ 20
  -> Chinchilla 70B + 1.4T tokens (D/N=20x) beats Gopher 280B

2023+ Overtraining (LLaMA 3, etc.):
  "Small model + massive data for cheaper inference"
  D/N >> 20, sometimes > 1000
  -> LLaMA 3 8B + 15T tokens (D/N=1875x)
```

Each shift overturned a piece of "common sense".


---

## Scaling Laws Checklist

Make sure you understand these (in order):

1. [ok] Power law: `Loss ∝ N^(-b)`; diminishing returns (doubling gives smaller and smaller absolute gains)
2. [ok] Kaplan: scale the model first; D grows slower than N
3. [ok] Chinchilla: model and data both matter; D/N ≈ 20
4. [ok] Overtraining: inference cost > training cost -> smaller model + more data can be optimal
5. [ok] µP: lets hyperparams transfer across model sizes; reduces tuning cost
6. [ok] FLOPs: `C ≈ 6PD` (forward ~2P + backward ~4P per token)
7. [ok] VRAM: `M ≈ 20P bytes` (params+grads+Adam+master+acts)
8. [ok] Rule of thumb: 1B params ~ 20GB training VRAM; inference ~2GB/B (no grads/optimizer/acts)

One-sentence summary: scaling laws tell you the best tradeoff (theory), FLOPs/VRAM estimates tell you whether you can afford it (engineering). In 2024, a common strategy is: smaller models + lots of data, optimizing for inference cost.
